In [4]:
import numpy as np
import pandas as pd
import PIL
import gc
import torch
import torch.nn as nn

import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models

In [5]:
class customYOLO(nn.Module):
    def __init__(self, in_channels=3, num_classes=5):
        """
            We implement a YOLO model from the original paper
            Input : Tensor of size (B, 448, 448, 3)
            Output : Tensor of size (B, 7, 7, 5)

            Each cell contains [x, y, w, h, confidence]
        """
        super(customYOLO, self).__init__()

        # Load pretrained resnet
        resnet = models.resnet50(pretrained=True)

        # Use resnet layers
        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4
        )

        self.detection_layers = nn.Sequential(
            nn.Conv2d(2048, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),
        )
        

        self.conv_head = nn.Sequential(
            nn.Conv2d(1024, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 5, kernel_size=1) # The 5 represents [x, y, w, h, conf]
        )


    # Pass the input through the model
    def forward(self, x):

        # Pass through resnet backbone
        x = self.backbone(x)

        # Pass through detection layers
        x = self.detection_layers(x)
        
        # Flatten before passing to mlp
        x = self.conv_head(x)

        # Reshape final output
        x = x.permute(0, 2, 3, 1)

        output = torch.sigmoid(x)
        
        return output


In [6]:
# Sample input for debugging
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device}")

x = torch.randn(3, 3, 448, 448).to(device)
model = customYOLO().to(device)
model.eval()

import time
start = time.time()
with torch.no_grad():
    out = model(x)
end = time.time()

print(f"Inference time: {(end - start)*1000:.2f} ms")
print(f"Output Shape: {out.shape}")

Using device cuda
Inference time: 6.75 ms
Output Shape: torch.Size([3, 7, 7, 5])


In [7]:



# Function to parse labels
def parse_file(path):
    with open(path) as f:
        lst = list(map(float, f.read().split()))

    # Initialize with zeros instead of empty
    out = torch.zeros(7, 7, 5)
    
    # Extract label values
    class_id = int(lst[0])
    center_x = lst[1]
    center_y = lst[2]
    width = lst[3]
    height = lst[4]

    # Find corresponding grid cell
    grid_x = int(center_x * S)
    grid_y = int(center_y * S)
    
    # Clamp to valid range
    grid_x = min(grid_x, S - 1)
    grid_y = min(grid_y, S - 1)

    # Calculate relative coords
    rel_x = center_x * S - grid_x
    rel_y = center_y * S - grid_y

    # Fill the grid cell
    out[grid_y, grid_x, 0] = rel_x
    out[grid_y, grid_x, 1] = rel_y
    out[grid_y, grid_x, 2] = width
    out[grid_y, grid_x, 3] = height
    out[grid_y, grid_x, 4] = 1.0  
    
    return out



In [8]:
# Test label parsing directly
import os

train_images_path = '/kaggle/input/combined-birds/combined-birds/train/images'
train_labels_path = '/kaggle/input/combined-birds/combined-birds/train/labels'


# Get a test file
test_img = os.listdir(train_images_path)[0]
print(f"\nTest image: {test_img}")

# Build label path
label_name = test_img.rsplit('.', 1)[0] + '.txt'
label_path = os.path.join(train_labels_path, label_name)
print(f"Label path: {label_path}")
print(f"Label exists: {os.path.exists(label_path)}")

# Read raw content
if os.path.exists(label_path):
    with open(label_path, 'r') as f:
        raw_content = f.read()
    print(f"\nRaw content: '{raw_content}'")
    print(f"Content length: {len(raw_content)}")
    print(f"Content repr: {repr(raw_content)}")
    
    # Try parsing
    try:
        values = list(map(float, raw_content.split()))
        print(f"\nParsed values: {values}")
        print(f"Number of values: {len(values)}")
        
        if len(values) >= 5:
            print(f"\nclass_id: {int(values[0])}")
            print(f"center_x: {values[1]}")
            print(f"center_y: {values[2]}")
            print(f"width: {values[3]}")
            print(f"height: {values[4]}")
            
            # Calculate grid position
            grid_x = int(values[1] * 7)
            grid_y = int(values[2] * 7)
            print(f"\nGrid position: ({grid_x}, {grid_y})")
            
            # Test full parse_file
            result = parse_file(label_path)
            print(f"\nParsed tensor non-zero cells: {(result.sum(dim=-1) > 0).sum().item()}")
            print(f"Cell [{grid_y}, {grid_x}] values: {result[grid_y, grid_x]}")
        else:
            print(f"\nERROR: Insufficient values in label file!")
    except Exception as e:
        print(f"\nERROR parsing: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\nERROR: Label file not found!")
    
    # List available labels
    all_labels = os.listdir(train_labels_path)[:5]
    print(f"\nFirst 5 label files: {all_labels}")


Test image: b1_1977.jpg
Label path: /kaggle/input/combined-birds/combined-birds/train/labels/b1_1977.txt
Label exists: True

Raw content: '0 0.624820 0.614865 0.604863 0.630221'
Content length: 37
Content repr: '0 0.624820 0.614865 0.604863 0.630221'

Parsed values: [0.0, 0.62482, 0.614865, 0.604863, 0.630221]
Number of values: 5

class_id: 0
center_x: 0.62482
center_y: 0.614865
width: 0.604863
height: 0.630221

Grid position: (4, 4)

ERROR parsing: name 'S' is not defined


Traceback (most recent call last):
  File "/tmp/ipykernel_55/2796610289.py", line 45, in <cell line: 0>
    result = parse_file(label_path)
             ^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_55/2150429747.py", line 17, in parse_file
    grid_x = int(center_x * S)
                            ^
NameError: name 'S' is not defined


In [9]:
# YOLO Loss Hyperparameters
LAMBDA_COORD = 10.0
LAMBDA_NOOBJ = 0.5
LAMBDA_OBJ = 2.0
S = 7



def loss_fn(preds, targets):
    # Extract components
    pred_xy = preds[..., :2]
    pred_hw = preds[..., 2:4]
    pred_conf = preds[..., 4]
    
    target_xy = targets[..., :2]
    target_hw = targets[..., 2:4]
    target_conf = targets[..., 4]

    # Create masks
    obj_mask = (target_conf > 0.5).unsqueeze(-1)
    obj_mask_conf = (target_conf > 0.5)
    noobj_mask = (target_conf <= 0.5)

    # Count cells per batch element
    num_obj = obj_mask_conf.float().sum() + 1e-6
    num_noobj = noobj_mask.float().sum() + 1e-6

    # Localization loss
    xy_loss = torch.sum(obj_mask * (pred_xy - target_xy) ** 2) / num_obj

    # Size loss
    hw_loss = torch.sum(obj_mask * (torch.sqrt(pred_hw + 1e-6) - torch.sqrt(target_hw + 1e-6)) ** 2) / num_obj

    # Confidence losses
    conf_obj_loss = LAMBDA_OBJ * torch.sum(obj_mask_conf * (pred_conf - target_conf)**2) / num_obj
    conf_noobj_loss = LAMBDA_NOOBJ * torch.sum(noobj_mask * (pred_conf)**2) / num_noobj

    # Combine
    total_loss = (
        LAMBDA_COORD * (xy_loss + hw_loss) +
        conf_obj_loss +
        conf_noobj_loss
    )
    
    return total_loss

In [10]:
import numpy as np
from collections import defaultdict

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes [x_center, y_center, width, height]
    All values are normalized between 0 and 1
    """
    # Convert to corner coordinates
    x1_min = box1[0] - box1[2] / 2
    y1_min = box1[1] - box1[3] / 2
    x1_max = box1[0] + box1[2] / 2
    y1_max = box1[1] + box1[3] / 2
    
    x2_min = box2[0] - box2[2] / 2
    y2_min = box2[1] - box2[3] / 2
    x2_max = box2[0] + box2[2] / 2
    y2_max = box2[1] + box2[3] / 2
    
    # Calculate intersection
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    inter_width = max(0, inter_x_max - inter_x_min)
    inter_height = max(0, inter_y_max - inter_y_min)
    intersection = inter_width * inter_height
    
    # Calculate union
    box1_area = box1[2] * box1[3]
    box2_area = box2[2] * box2[3]
    union = box1_area + box2_area - intersection
    
    if union == 0:
        return 0
    
    return intersection / union


In [11]:
def get_predictions_from_grid(predictions, confidence_threshold=0.5):
    """
    Convert grid predictions to list of bounding boxes
    predictions: (B, 7, 7, 5) tensor
    Returns: list of predictions per image
    """
    batch_size = predictions.shape[0]
    all_predictions = []
    
    for b in range(batch_size):
        img_predictions = []
        for i in range(7):
            for j in range(7):
                conf = predictions[b, i, j, 4].item()
                if conf > confidence_threshold:
                    # Convert grid cell coordinates to image coordinates
                    x_cell = predictions[b, i, j, 0].item()
                    y_cell = predictions[b, i, j, 1].item()
                    w = predictions[b, i, j, 2].item()
                    h = predictions[b, i, j, 3].item()
                    
                    # Convert to absolute coordinates (0-1 range)
                    x_center = (j + x_cell) / 7.0
                    y_center = (i + y_cell) / 7.0
                    
                    img_predictions.append({
                        'box': [x_center, y_center, w, h],
                        'confidence': conf
                    })
        all_predictions.append(img_predictions)
    
    return all_predictions

In [12]:
def get_targets_from_grid(targets):
    """
    Convert grid targets to list of bounding boxes
    targets: (B, 7, 7, 5) tensor
    Returns: list of ground truth boxes per image
    """
    batch_size = targets.shape[0]
    all_targets = []
    
    for b in range(batch_size):
        img_targets = []
        for i in range(7):
            for j in range(7):
                if targets[b, i, j, 4].item() > 0.5:  # Has object
                    x_cell = targets[b, i, j, 0].item()
                    y_cell = targets[b, i, j, 1].item()
                    w = targets[b, i, j, 2].item()
                    h = targets[b, i, j, 3].item()
                    
                    # Convert to absolute coordinates
                    x_center = (j + x_cell) / 7.0
                    y_center = (i + y_cell) / 7.0
                    
                    img_targets.append([x_center, y_center, w, h])
        all_targets.append(img_targets)
    
    return all_targets


In [13]:
def calculate_ap(precisions, recalls):
    """Calculate Average Precision using 11-point interpolation"""
    ap = 0
    for t in np.linspace(0, 1, 11):
        if np.sum(recalls >= t) == 0:
            p = 0
        else:
            p = np.max(precisions[recalls >= t])
        ap += p / 11
    return ap

def calculate_metrics(predictions, targets, iou_threshold=0.5):
    """
    Calculate precision, recall, and mAP
    predictions: list of predictions per image
    targets: list of ground truth boxes per image
    """
    all_pred_boxes = []
    all_pred_confs = []
    all_gt_boxes = []
    
    # Collect all predictions and ground truths
    for img_idx, (preds, gts) in enumerate(zip(predictions, targets)):
        for pred in preds:
            all_pred_boxes.append((img_idx, pred['box']))
            all_pred_confs.append(pred['confidence'])
        
        for gt in gts:
            all_gt_boxes.append((img_idx, gt))
    
    # Sort predictions by confidence
    sorted_indices = np.argsort(all_pred_confs)[::-1]
    
    tp = np.zeros(len(sorted_indices))
    fp = np.zeros(len(sorted_indices))
    
    # Track which ground truths have been matched
    matched_gt = set()
    
    for pred_idx, sorted_idx in enumerate(sorted_indices):
        img_idx, pred_box = all_pred_boxes[sorted_idx]
        
        # Find matching ground truth
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, (gt_img_idx, gt_box) in enumerate(all_gt_boxes):
            if gt_img_idx != img_idx or gt_idx in matched_gt:
                continue
            
            iou = calculate_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold and best_gt_idx not in matched_gt:
            tp[pred_idx] = 1
            matched_gt.add(best_gt_idx)
        else:
            fp[pred_idx] = 1
    
    # Calculate cumulative precision and recall
    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)
    
    recalls = tp_cumsum / max(len(all_gt_boxes), 1)
    precisions = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1e-6)
    
    # Calculate AP
    ap = calculate_ap(precisions, recalls)
    
    # Final precision and recall
    final_precision = precisions[-1] if len(precisions) > 0 else 0
    final_recall = recalls[-1] if len(recalls) > 0 else 0
    
    return {
        'precision': final_precision,
        'recall': final_recall,
        'ap': ap,
        'precisions': precisions,
        'recalls': recalls
    }


In [14]:
def evaluate_model(model, dataloader, device, confidence_threshold=0.5, iou_threshold=0.5):
    """
    Evaluate model on a dataset
    Returns metrics dictionary
    """
    model.eval()
    
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            targets = targets.to(device)
            
            predictions = model(images)
            
            # Convert to lists of boxes
            batch_preds = get_predictions_from_grid(predictions, confidence_threshold)
            batch_targets = get_targets_from_grid(targets)
            
            all_predictions.extend(batch_preds)
            all_targets.extend(batch_targets)
    
    # Calculate metrics
    metrics = calculate_metrics(all_predictions, all_targets, iou_threshold)
    
    return metrics


In [ ]:
import os
import time
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split

# --- 1. MEMORY EFFICIENT DATASET ---
class BirdDataset(Dataset):
    def __init__(self, images_dir, labels_dir, image_files, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = image_files
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        # Load image on the fly (Saves RAM)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Load label
        label_name = img_name.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(self.labels_dir, label_name)
        
        try:
            # Assuming parse_file is defined in your environment
            target = parse_file(label_path) 
        except:
            target = torch.zeros(7, 7, 5) # Fallback
            
        return image, target

# --- 2. AUGMENTATION & HYPERPARAMETERS ---
LEARNING_RATE = 5e-4 # Lower starting rate for stability
WEIGHT_DECAY = 1e-2  # Standard for AdamW
BATCH_SIZE = 32      # Increased for better gradient stability
N_EPOCHS = 50
WARMUP_EPOCHS = 3

# High-quality augmentations to prevent overfitting
train_transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 3. PREPARE DATA ---
all_images = sorted([f for f in os.listdir(train_images_path) if f.endswith(('.jpg', '.png'))])
train_files, val_files = train_test_split(all_images, test_size=0.15, random_state=42)

train_dataset = BirdDataset(train_images_path, train_labels_path, train_files, transform=train_transform)
val_dataset = BirdDataset(train_images_path, train_labels_path, val_files, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# --- 4. MODEL SETUP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = customYOLO().to(device)

# Initial Freeze
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning Rate Schedulers
# Warmup + Cosine Decay is the industry standard for single-run optimization
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)
    return 1.0

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS - WARMUP_EPOCHS)

# --- 5. TRAINING LOOP WITH PROGRESSION ---
best_val_map = 0.0

for epoch in range(N_EPOCHS):
    # Unfreeze backbone halfway through to fine-tune
    # if epoch == N_EPOCHS // 2:
    #     print("--- Unfreezing Backbone for Fine-Tuning ---")
    #     for param in model.backbone.parameters():
    #         param.requires_grad = True
    #     # Lower LR for fine-tuning
    #     for param_group in optimizer.param_groups:
    #         param_group['lr'] /= 10

    model.train()
    total_train_loss = 0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        predictions = model(images)
        loss = loss_fn(predictions, targets)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0) # Tighter clip
        optimizer.step()
        
        total_train_loss += loss.item()

    # Step schedulers
    if epoch < WARMUP_EPOCHS:
        warmup_scheduler.step()
    else:
        cosine_scheduler.step()

    # VALIDATION & METRICS
    model.eval()
    val_metrics = evaluate_model(model, val_loader, device) # Using your eval function
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{N_EPOCHS} | Loss: {total_train_loss/len(train_loader):.4f} | mAP: {val_metrics['ap']:.4f} | LR: {current_lr:.6f}")

    # Save Best
    if val_metrics['ap'] > best_val_map:
        best_val_map = val_metrics['ap']
        torch.save(model.state_dict(), 'best_bird_model.pth')
        print("--> Model Saved!")

    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/50 | Loss: 2.0131 | mAP: 0.2598 | LR: 0.000333
--> Model Saved!
Epoch 2/50 | Loss: 1.3679 | mAP: 0.3858 | LR: 0.000500
--> Model Saved!
Epoch 3/50 | Loss: 1.2295 | mAP: 0.4105 | LR: 0.000500
--> Model Saved!
Epoch 4/50 | Loss: 1.0800 | mAP: 0.2917 | LR: 0.000499
Epoch 5/50 | Loss: 0.9656 | mAP: 0.4163 | LR: 0.000498
--> Model Saved!
Epoch 6/50 | Loss: 0.8471 | mAP: 0.4764 | LR: 0.000495
--> Model Saved!
Epoch 7/50 | Loss: 0.7538 | mAP: 0.4338 | LR: 0.000491
Epoch 8/50 | Loss: 0.6715 | mAP: 0.3535 | LR: 0.000486
Epoch 9/50 | Loss: 0.6211 | mAP: 0.4309 | LR: 0.000480
Epoch 10/50 | Loss: 0.5026 | mAP: 0.4128 | LR: 0.000473
Epoch 11/50 | Loss: 0.4590 | mAP: 0.4922 | LR: 0.000465
--> Model Saved!
Epoch 12/50 | Loss: 0.3942 | mAP: 0.4086 | LR: 0.000456
Epoch 13/50 | Loss: 0.3663 | mAP: 0.5367 | LR: 0.000446
--> Model Saved!
Epoch 14/50 | Loss: 0.3109 | mAP: 0.5625 | LR: 0.000435
--> Model Saved!
Epoch 15/50 | Loss: 0.3063 | mAP: 0.4751 | LR: 0.000424
Epoch 16/50 | Loss: 0.2687 | mAP:

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss curves
axes[0, 0].plot(train_losses, label='Train Loss')
axes[0, 0].plot(val_losses, label='Val Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# mAP curve
val_maps = [m['ap'] for m in val_metrics_history]
axes[0, 1].plot(val_maps, label='Val mAP', color='green')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('mAP')
axes[0, 1].set_title('Validation mAP')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision curve
val_precisions = [m['precision'] for m in val_metrics_history]
axes[1, 0].plot(val_precisions, label='Val Precision', color='blue')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Validation Precision')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Recall curve
val_recalls = [m['recall'] for m in val_metrics_history]
axes[1, 1].plot(val_recalls, label='Val Recall', color='red')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_title('Validation Recall')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_metrics.png', dpi=150)
plt.show()

# Precision-Recall curve for final epoch
final_metrics = val_metrics_history[-1]
plt.figure(figsize=(8, 6))
plt.plot(final_metrics['recalls'], final_metrics['precisions'], linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (mAP: {final_metrics["ap"]:.4f})')
plt.grid(True)
plt.savefig('/kaggle/working/precision_recall_curve.png', dpi=150)
plt.show()

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import matplotlib.pyplot as plt

# Data extracted from your training logs
epochs = list(range(1, 51))
losses = [6.2085, 4.9053, 4.5015, 4.0267, 3.6750, 3.6586, 3.2419, 3.1979, 3.0340, 3.1231, 
          2.8378, 2.7476, 2.4937, 2.3514, 2.3012, 2.2690, 2.3487, 2.1815, 2.0404, 2.0851, 
          2.0147, 1.9723, 2.0032, 1.7871, 1.5902, 1.7122, 1.6086, 1.7638, 1.5311, 1.3216, 
          1.2937, 1.3194, 1.2007, 1.0833, 1.1757, 1.0817, 0.9253, 0.9877, 1.0790, 1.0110, 
          0.9267, 0.8486, 0.8852, 0.8417, 0.8551, 0.8845, 0.8912, 0.7985, 0.8927, 0.9535]

maps = [0.0445, 0.0873, 0.1089, 0.1584, 0.1545, 0.2196, 0.1633, 0.0969, 0.2460, 0.1342, 
        0.1719, 0.1639, 0.2016, 0.1760, 0.2570, 0.2599, 0.2730, 0.2759, 0.2889, 0.1810, 
        0.2686, 0.2573, 0.2745, 0.2188, 0.2492, 0.3405, 0.2761, 0.2383, 0.2516, 0.2437, 
        0.2587, 0.3040, 0.2626, 0.3163, 0.2662, 0.3246, 0.3188, 0.3082, 0.2478, 0.2559, 
        0.3187, 0.3268, 0.3184, 0.3167, 0.3204, 0.3283, 0.3269, 0.3251, 0.3277, 0.3305]

# Style configuration
plt.style.use('seaborn-v0_8-muted') 
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# 1. Plot Training Loss
ax1.plot(epochs, losses, color='#e74c3c', linewidth=2.5, label='Total Loss')
ax1.set_title('Model Convergence (Training Loss)', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel('Loss Value', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend(loc='upper right')

# Add a marker for the "best" mAP epoch on the loss curve
best_epoch = maps.index(max(maps)) + 1
# 2. Plot mAP (Mean Average Precision)
ax2.plot(epochs, maps, color='#2ecc71', linewidth=2.5, marker='o', markersize=4, label='mAP @ 0.5 IoU')
ax2.fill_between(epochs, maps, color='#2ecc71', alpha=0.1) # Shaded area for visual weight
ax2.set_title('Object Detection Accuracy (mAP)', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('mAP Score', fontsize=12)
ax2.set_ylim(0, 0.4) # Adjust based on your max mAP
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()
plt.savefig('/kaggle/working/metrics.png')